# 1. Step up and installation

### Evaluation metric

Kaggle uses the **Brier score** (mean squared error between predicted probability and outcome):

$$\text{Brier} = \frac{1}{N}\sum_{i=1}^{N}(p_i - y_i)^2$$

Unlike log-loss, the Brier score does **not** explode for confident wrong predictions — the maximum penalty per game is 1.0. This makes extreme predictions (e.g., 0.99 for a 1-seed vs 16-seed) less risky than under log-loss. Nonetheless, we still clip predictions to [0.01, 0.99] and invest in calibration (Section 8) to minimize overall error.

In [1]:
# Python libraries 
import numpy as np 
import matplotlib.pyplot as plt
import pandas as pd 

import os


# 2. Load Dataset 


I load **Detailed Results** files (box scores) for both men's and women's basketball.

| File | Contents | Available from |
|------|----------|---------------|
| `MRegularSeasonDetailedResults` | Every men's regular-season game with full box scores | 2003 |
| `WRegularSeasonDetailedResults` | Every women's regular-season game with full box scores | 2010 |
| `M/WNCAATourneyDetailedResults` | Tournament games with full box scores | same |
| `M/WNCAATourneySeeds` | Seedings (1–16) for each tournament team | same |

**Box score columns** include FGM/FGA (field goals made/attempted), FGM3/FGA3 (three-pointers), FTM/FTA (free throws), OR/DR (offensive/defensive rebounds), Ast (assists), TO (turnovers), Stl (steals), Blk (blocks), PF (personal fouls).

Data is current through early February 2026. The remaining regular-season weeks will be added before Selection Sunday (March 15).



In [2]:
# Locate directory 
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/march-machine-learning-mania-2026/Conferences.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/WNCAATourneyDetailedResults.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/WRegularSeasonCompactResults.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/MNCAATourneySeedRoundSlots.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/MRegularSeasonDetailedResults.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/MNCAATourneyCompactResults.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/MGameCities.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/WSecondaryTourneyCompactResults.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/WGameCities.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/MSeasons.csv
/kaggle/input/competitions/march-machine-learning-mania-2026/WNCAATourneySlots.csv
/kaggle/input/competitions/march-machine-learning

In [3]:
# Load all the dataset 
directPath = '/kaggle/input/competitions/march-machine-learning-mania-2026'

dataframes = {}

for file in os.listdir(directPath):
    if file.endswith(".csv"):
        name = file.replace(".csv", "")
        dataframes[name] = pd.read_csv(os.path.join(directPath, file))

print(dataframes.keys())



dict_keys(['Conferences', 'WNCAATourneyDetailedResults', 'WRegularSeasonCompactResults', 'MNCAATourneySeedRoundSlots', 'MRegularSeasonDetailedResults', 'MNCAATourneyCompactResults', 'MGameCities', 'WSecondaryTourneyCompactResults', 'WGameCities', 'MSeasons', 'WNCAATourneySlots', 'MSecondaryTourneyTeams', 'SampleSubmissionStage2', 'Cities', 'MTeamSpellings', 'MRegularSeasonCompactResults', 'MMasseyOrdinals', 'MSecondaryTourneyCompactResults', 'WTeams', 'WConferenceTourneyGames', 'MNCAATourneySlots', 'MNCAATourneySeeds', 'WNCAATourneyCompactResults', 'WSeasons', 'WNCAATourneySeeds', 'MTeamCoaches', 'MConferenceTourneyGames', 'WRegularSeasonDetailedResults', 'MNCAATourneyDetailedResults', 'WTeamSpellings', 'MTeamConferences', 'MTeams', 'WTeamConferences', 'SampleSubmissionStage1', 'WSecondaryTourneyTeams'])


In [4]:
#  Men's basketball
mensregularResults = pd.read_csv(f"{directPath}/MRegularSeasonDetailedResults.csv")
menstourneyResults = pd.read_csv(f"{directPath}/MNCAATourneyDetailedResults.csv")
MenSeeds = pd.read_csv(f"{directPath}/MNCAATourneySeeds.csv")

# Women's basketball
womensregularResults = pd.read_csv(f"{directPath}/WRegularSeasonDetailedResults.csv")
womenstourneyResults = pd.read_csv(f"{directPath}/WNCAATourneyDetailedResults.csv")
WomenSeeds = pd.read_csv(f"{directPath}/WNCAATourneySeeds.csv")